# FITE7405 Assignment 3: Mini Option Pricer

这个 notebook 按照作业要求实现了主要代码部分，并且把每个模块拆成了清晰的 Markdown + Code 单元，便于直接展示和提交。

已实现内容：

1. Black-Scholes 欧式期权定价
2. 隐含波动率反推
3. 几何平均 Asian / Basket 闭式解
4. 算术平均 Asian / Basket 的 Monte Carlo 与 Control Variate
5. KIKO Put 的 Quasi-Monte Carlo 定价与 Delta
6. 美式期权 Binomial Tree
7. 一个基于 `ipywidgets` 的 notebook 内部小型 GUI

说明：

- Monte Carlo 与 Quasi-Monte Carlo 都使用固定随机种子，保证结果可复现。
- Asian 和 Basket 部分已经按题目给出的测试参数准备了批量测试单元。
- GUI 单元里会展示所有输入框，不相关的字段会被自动忽略。


In [1]:
!nvidia-smi

Mon Mar 30 16:51:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Imports And Numeric Utilities

先准备所有后续模块都会用到的基础函数，包括：

- 标准正态分布的 CDF 与近似反函数
- 统一的 `call / put` 支付函数
- Monte Carlo 结果摘要函数（价格、标准误、95% 置信区间）


In [ ]:
from math import erf, exp, log, sqrt

import numpy as np


DEFAULT_SEED = 42


def validate_option_type(option_type):
    option_type = option_type.lower().strip()
    if option_type not in {"call", "put"}:
        raise ValueError("option_type must be either 'call' or 'put'.")
    return option_type


def option_payoff(values, strike, option_type):
    option_type = validate_option_type(option_type)
    values = np.asarray(values, dtype=float)
    if option_type == "call":
        return np.maximum(values - strike, 0.0)
    return np.maximum(strike - values, 0.0)


def scalar_payoff(value, strike, option_type):
    option_type = validate_option_type(option_type)
    if option_type == "call":
        return max(value - strike, 0.0)
    return max(strike - value, 0.0)


def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))


def norm_ppf(p):
    p = np.asarray(p, dtype=float)
    if np.any((p <= 0.0) | (p >= 1.0)):
        raise ValueError("Probabilities must be strictly inside (0, 1).")

    a = np.array(
        [
            -3.969683028665376e01,
            2.209460984245205e02,
            -2.759285104469687e02,
            1.383577518672690e02,
            -3.066479806614716e01,
            2.506628277459239e00,
        ]
    )
    b = np.array(
        [
            -5.447609879822406e01,
            1.615858368580409e02,
            -1.556989798598866e02,
            6.680131188771972e01,
            -1.328068155288572e01,
        ]
    )
    c = np.array(
        [
            -7.784894002430293e-03,
            -3.223964580411365e-01,
            -2.400758277161838e00,
            -2.549732539343734e00,
            4.374664141464968e00,
            2.938163982698783e00,
        ]
    )
    d = np.array(
        [
            7.784695709041462e-03,
            3.224671290700398e-01,
            2.445134137142996e00,
            3.754408661907416e00,
        ]
    )

    plow = 0.02425
    phigh = 1.0 - plow
    x = np.empty_like(p)

    lower = p < plow
    upper = p > phigh
    central = ~(lower | upper)

    if np.any(lower):
        q = np.sqrt(-2.0 * np.log(p[lower]))
        num = (((((c[0] * q + c[1]) * q + c[2]) * q + c[3]) * q + c[4]) * q + c[5])
        den = ((((d[0] * q + d[1]) * q + d[2]) * q + d[3]) * q + 1.0)
        x[lower] = num / den

    if np.any(upper):
        q = np.sqrt(-2.0 * np.log(1.0 - p[upper]))
        num = (((((c[0] * q + c[1]) * q + c[2]) * q + c[3]) * q + c[4]) * q + c[5])
        den = ((((d[0] * q + d[1]) * q + d[2]) * q + d[3]) * q + 1.0)
        x[upper] = -(num / den)

    if np.any(central):
        q = p[central] - 0.5
        r = q * q
        num = (((((a[0] * r + a[1]) * r + a[2]) * r + a[3]) * r + a[4]) * r + a[5]) * q
        den = (((((b[0] * r + b[1]) * r + b[2]) * r + b[3]) * r + b[4]) * r + 1.0)
        x[central] = num / den

    return x


def summarize_discounted_payoffs(samples):
    samples = np.asarray(samples, dtype=float)
    if samples.size == 0:
        raise ValueError("At least one sample is required.")

    price = float(np.mean(samples))
    if samples.size == 1:
        stderr = 0.0
    else:
        stderr = float(np.std(samples, ddof=1) / sqrt(samples.size))

    return {
        "price": price,
        "stderr": stderr,
        "ci95": (price - 1.96 * stderr, price + 1.96 * stderr),
    }


## 2. European Options And Implied Volatility

这一部分实现题目要求的：

- Black-Scholes 欧式看涨 / 看跌期权价格
- 在给定 market premium 的情况下用二分法反推隐含波动率

欧式部分额外支持 repo rate `q`，和题目输入格式保持一致。


In [ ]:
def black_scholes_price(s0, sigma, r, T, K, option_type="call", q=0.0):
    option_type = validate_option_type(option_type)

    if T <= 0.0 or sigma <= 0.0:
        forward_spot = s0 * exp((r - q) * T)
        return exp(-r * T) * scalar_payoff(forward_spot, K, option_type)

    sqrt_T = sqrt(T)
    d1 = (log(s0 / K) + (r - q + 0.5 * sigma * sigma) * T) / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T

    discounted_spot = s0 * exp(-q * T)
    discounted_strike = K * exp(-r * T)

    if option_type == "call":
        return discounted_spot * norm_cdf(d1) - discounted_strike * norm_cdf(d2)
    return discounted_strike * norm_cdf(-d2) - discounted_spot * norm_cdf(-d1)


def implied_volatility(market_price, s0, r, q, T, K, option_type="call", tol=1e-8, max_iter=200):
    option_type = validate_option_type(option_type)
    if market_price <= 0.0:
        return 0.0
    if T <= 0.0:
        raise ValueError("Implied volatility is undefined when T <= 0.")

    discounted_spot = s0 * exp(-q * T)
    discounted_strike = K * exp(-r * T)
    if option_type == "call":
        lower_bound = max(0.0, discounted_spot - discounted_strike)
        upper_bound = discounted_spot
    else:
        lower_bound = max(0.0, discounted_strike - discounted_spot)
        upper_bound = discounted_strike

    if not (lower_bound - tol <= market_price <= upper_bound + tol):
        raise ValueError("Market price is outside the no-arbitrage bounds.")

    low, high = 1e-8, 1.0
    while black_scholes_price(s0, high, r, T, K, option_type, q) < market_price and high < 10.0:
        high *= 2.0

    for _ in range(max_iter):
        mid = 0.5 * (low + high)
        model_price = black_scholes_price(s0, mid, r, T, K, option_type, q)
        if abs(model_price - market_price) < tol:
            return mid
        if model_price < market_price:
            low = mid
        else:
            high = mid

    return 0.5 * (low + high)


## 3. Asian Options

这里分成两部分：

- 几何平均 Asian option：使用闭式解
- 算术平均 Asian option：使用 Monte Carlo，并支持用几何平均 Asian option 作为 control variate

观测时点按照题目要求取：

`T / n, 2T / n, ..., T`


In [ ]:
def geometric_asian_option_price(s0, sigma, r, T, K, n_obs, option_type="call"):
    option_type = validate_option_type(option_type)
    if n_obs < 1:
        raise ValueError("n_obs must be at least 1.")

    mean_log = log(s0) + (r - 0.5 * sigma * sigma) * T * (n_obs + 1) / (2.0 * n_obs)
    var_log = sigma * sigma * T * (n_obs + 1) * (2 * n_obs + 1) / (6.0 * n_obs * n_obs)

    if var_log <= 1e-14:
        deterministic_average = exp(mean_log)
        return exp(-r * T) * scalar_payoff(deterministic_average, K, option_type)

    std_log = sqrt(var_log)
    d1 = (mean_log - log(K) + var_log) / std_log
    d2 = d1 - std_log
    expected_average = exp(mean_log + 0.5 * var_log)

    if option_type == "call":
        return exp(-r * T) * (expected_average * norm_cdf(d1) - K * norm_cdf(d2))
    return exp(-r * T) * (K * norm_cdf(-d2) - expected_average * norm_cdf(-d1))


def simulate_gbm_paths(s0, sigma, r, T, n_obs, n_paths, seed=DEFAULT_SEED):
    if n_obs < 1:
        raise ValueError("n_obs must be at least 1.")
    if n_paths < 1:
        raise ValueError("n_paths must be at least 1.")

    dt = T / n_obs
    rng = np.random.default_rng(seed)
    z = rng.standard_normal((n_paths, n_obs))
    log_paths = np.log(s0) + np.cumsum((r - 0.5 * sigma * sigma) * dt + sigma * np.sqrt(dt) * z, axis=1)
    return np.exp(log_paths)


def arithmetic_asian_option_mc(
    s0,
    sigma,
    r,
    T,
    K,
    n_obs,
    option_type="call",
    n_paths=100_000,
    control_variate="none",
    seed=DEFAULT_SEED,
):
    option_type = validate_option_type(option_type)
    control_variate = control_variate.lower().strip()

    paths = simulate_gbm_paths(s0, sigma, r, T, n_obs, n_paths, seed)
    arithmetic_average = np.mean(paths, axis=1)
    geometric_average = np.exp(np.mean(np.log(paths), axis=1))

    discounted_arithmetic = exp(-r * T) * option_payoff(arithmetic_average, K, option_type)
    discounted_geometric = exp(-r * T) * option_payoff(geometric_average, K, option_type)

    beta = 0.0
    if control_variate in {"none", "no", "no control variate"}:
        adjusted_payoffs = discounted_arithmetic
    elif control_variate in {"geometric", "geometric asian"}:
        exact_geometric_price = geometric_asian_option_price(s0, sigma, r, T, K, n_obs, option_type)
        variance = float(np.var(discounted_geometric, ddof=1))
        if variance > 0.0:
            covariance = float(np.cov(discounted_arithmetic, discounted_geometric, ddof=1)[0, 1])
            beta = covariance / variance
        adjusted_payoffs = discounted_arithmetic - beta * (discounted_geometric - exact_geometric_price)
    else:
        raise ValueError("control_variate must be 'none' or 'geometric'.")

    result = summarize_discounted_payoffs(adjusted_payoffs)
    result["control_variate"] = control_variate
    result["beta"] = beta
    return result


## 4. Basket Options

Basket 部分按照题目要求只考虑两个标的资产：

- 几何平均 Basket option：闭式解
- 算术平均 Basket option：Monte Carlo
- 对算术平均 Basket option，用几何平均 Basket option 做 control variate


In [ ]:
def geometric_basket_option_price(s1, s2, sigma1, sigma2, r, T, K, rho, option_type="call"):
    option_type = validate_option_type(option_type)
    if not -1.0 <= rho <= 1.0:
        raise ValueError("rho must be between -1 and 1.")

    mean_log = 0.5 * (log(s1) + log(s2)) + (r - 0.25 * (sigma1 * sigma1 + sigma2 * sigma2)) * T
    var_log = 0.25 * (sigma1 * sigma1 + sigma2 * sigma2 + 2.0 * rho * sigma1 * sigma2) * T

    if var_log <= 1e-14:
        deterministic_average = exp(mean_log)
        return exp(-r * T) * scalar_payoff(deterministic_average, K, option_type)

    std_log = sqrt(var_log)
    d1 = (mean_log - log(K) + var_log) / std_log
    d2 = d1 - std_log
    expected_average = exp(mean_log + 0.5 * var_log)

    if option_type == "call":
        return exp(-r * T) * (expected_average * norm_cdf(d1) - K * norm_cdf(d2))
    return exp(-r * T) * (K * norm_cdf(-d2) - expected_average * norm_cdf(-d1))


def arithmetic_basket_option_mc(
    s1,
    s2,
    sigma1,
    sigma2,
    r,
    T,
    K,
    rho,
    option_type="call",
    n_paths=100_000,
    control_variate="none",
    seed=DEFAULT_SEED,
):
    option_type = validate_option_type(option_type)
    control_variate = control_variate.lower().strip()
    if not -1.0 <= rho <= 1.0:
        raise ValueError("rho must be between -1 and 1.")

    rng = np.random.default_rng(seed)
    z1 = rng.standard_normal(n_paths)
    z2 = rng.standard_normal(n_paths)
    z2 = rho * z1 + sqrt(1.0 - rho * rho) * z2

    s1_T = s1 * np.exp((r - 0.5 * sigma1 * sigma1) * T + sigma1 * sqrt(T) * z1)
    s2_T = s2 * np.exp((r - 0.5 * sigma2 * sigma2) * T + sigma2 * sqrt(T) * z2)

    arithmetic_average = 0.5 * (s1_T + s2_T)
    geometric_average = np.sqrt(s1_T * s2_T)

    discounted_arithmetic = exp(-r * T) * option_payoff(arithmetic_average, K, option_type)
    discounted_geometric = exp(-r * T) * option_payoff(geometric_average, K, option_type)

    beta = 0.0
    if control_variate in {"none", "no", "no control variate"}:
        adjusted_payoffs = discounted_arithmetic
    elif control_variate in {"geometric", "geometric basket"}:
        exact_geometric_price = geometric_basket_option_price(
            s1, s2, sigma1, sigma2, r, T, K, rho, option_type
        )
        variance = float(np.var(discounted_geometric, ddof=1))
        if variance > 0.0:
            covariance = float(np.cov(discounted_arithmetic, discounted_geometric, ddof=1)[0, 1])
            beta = covariance / variance
        adjusted_payoffs = discounted_arithmetic - beta * (discounted_geometric - exact_geometric_price)
    else:
        raise ValueError("control_variate must be 'none' or 'geometric'.")

    result = summarize_discounted_payoffs(adjusted_payoffs)
    result["control_variate"] = control_variate
    result["beta"] = beta
    return result


## 5. KIKO Put With Rebate Via Quasi-Monte Carlo

这一部分实现：

- KIKO put 的路径模拟
- 用随机平移的 Halton sequence 做 quasi-Monte Carlo
- 用 central difference + common random numbers 计算 Delta

由于题目只要求价格和 Delta，这里额外返回了 batch-based 的标准误，方便检查数值稳定性。


In [ ]:
def first_n_primes(n):
    primes = []
    candidate = 2
    while len(primes) < n:
        is_prime = True
        limit = int(sqrt(candidate))
        for prime in primes:
            if prime > limit:
                break
            if candidate % prime == 0:
                is_prime = False
                break
        if is_prime:
            primes.append(candidate)
        candidate += 1
    return primes


def van_der_corput(size, base, start_index=0):
    sequence = np.empty(size, dtype=float)
    for i in range(size):
        index = i + start_index + 1
        factor = 1.0 / base
        value = 0.0
        while index > 0:
            index, remainder = divmod(index, base)
            value += remainder * factor
            factor /= base
        sequence[i] = value
    return sequence


def halton_sequence(size, dim, skip=32):
    primes = first_n_primes(dim)
    return np.column_stack([van_der_corput(size, base, skip) for base in primes])


def kiko_put_discounted_payoffs_from_points(points, s0, sigma, r, T, K, L, U, n_obs, rebate):
    if n_obs < 1:
        raise ValueError("n_obs must be at least 1.")
    if not (L < s0 < U):
        raise ValueError("The barriers must satisfy L < S0 < U.")

    z = norm_ppf(np.clip(points, 1e-12, 1.0 - 1e-12))
    dt = T / n_obs
    drift = (r - 0.5 * sigma * sigma) * dt
    diffusion = sigma * sqrt(dt)
    discount_to_maturity = exp(-r * T)

    spots = np.full(points.shape[0], s0, dtype=float)
    knocked_in = np.zeros(points.shape[0], dtype=bool)
    active = np.ones(points.shape[0], dtype=bool)
    discounted_payoffs = np.zeros(points.shape[0], dtype=float)

    for step in range(n_obs):
        spots[active] *= np.exp(drift + diffusion * z[active, step])
        current_time = (step + 1) * dt

        knocked_out = active & (spots >= U)
        if np.any(knocked_out):
            discounted_payoffs[knocked_out] = rebate * exp(-r * current_time)

        active = active & (~knocked_out)
        knocked_in = knocked_in | (active & (spots <= L))

    if np.any(active):
        discounted_payoffs[active] = (
            discount_to_maturity
            * option_payoff(spots[active], K, "put")
            * knocked_in[active]
        )

    return discounted_payoffs


def quasi_mc_kiko_put(
    s0,
    sigma,
    r,
    T,
    K,
    L,
    U,
    n_obs,
    rebate,
    n_paths=8192,
    n_batches=8,
    seed=DEFAULT_SEED,
    bump=None,
):
    if bump is None:
        bump = max(0.5, 0.01 * s0)
    if n_batches < 1:
        raise ValueError("n_batches must be at least 1.")

    batch_size = max(1, n_paths // n_batches)
    rng = np.random.default_rng(seed)
    base_points = halton_sequence(batch_size, n_obs, skip=32)

    price_estimates = []
    delta_estimates = []

    for shift in rng.random((n_batches, n_obs)):
        points = (base_points + shift) % 1.0

        mid = kiko_put_discounted_payoffs_from_points(
            points, s0, sigma, r, T, K, L, U, n_obs, rebate
        ).mean()
        up = kiko_put_discounted_payoffs_from_points(
            points, s0 + bump, sigma, r, T, K, L, U, n_obs, rebate
        ).mean()
        down = kiko_put_discounted_payoffs_from_points(
            points, max(1e-8, s0 - bump), sigma, r, T, K, L, U, n_obs, rebate
        ).mean()

        price_estimates.append(mid)
        delta_estimates.append((up - down) / (2.0 * bump))

    price_estimates = np.asarray(price_estimates, dtype=float)
    delta_estimates = np.asarray(delta_estimates, dtype=float)

    return {
        "price": float(np.mean(price_estimates)),
        "price_stderr": float(np.std(price_estimates, ddof=1) / sqrt(len(price_estimates))),
        "delta": float(np.mean(delta_estimates)),
        "delta_stderr": float(np.std(delta_estimates, ddof=1) / sqrt(len(delta_estimates))),
        "effective_paths": int(batch_size * n_batches),
        "bump": float(bump),
    }


## 6. American Options With Binomial Tree

最后一类必做定价模型是美式期权。这里采用标准的 Cox-Ross-Rubinstein 二叉树，并在每一个节点比较：

- 继续持有价值
- 立即行权价值

从而得到美式 call / put 的价格。


In [ ]:
def american_option_binomial(s0, sigma, r, T, K, steps, option_type="call", q=0.0):
    option_type = validate_option_type(option_type)
    if steps < 1:
        raise ValueError("steps must be at least 1.")

    dt = T / steps
    u = exp(sigma * sqrt(dt))
    d = 1.0 / u
    growth = exp((r - q) * dt)
    p = (growth - d) / (u - d)

    if not (0.0 <= p <= 1.0):
        raise ValueError("Risk-neutral probability is outside [0, 1].")

    discount = exp(-r * dt)
    j = np.arange(steps + 1)
    spots = s0 * (u ** j) * (d ** (steps - j))
    values = option_payoff(spots, K, option_type)

    for step in range(steps - 1, -1, -1):
        j = np.arange(step + 1)
        spots = s0 * (u ** j) * (d ** (step - j))
        continuation = discount * (p * values[1:] + (1.0 - p) * values[:-1])
        exercise = option_payoff(spots, K, option_type)
        values = np.maximum(continuation, exercise)

    return float(values[0])


## 7. Test Helpers

下面先定义一个简单的表格输出函数，再把题目要求的 Asian / Basket 测试批量跑出来。

这样做的好处是：

- 可以直接把运行结果复制到报告里
- 可以直观看到 control variate 是否明显缩小了置信区间


In [ ]:
def format_value(value, digits=6):
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.{digits}f}"
    if isinstance(value, tuple) and len(value) == 2:
        left, right = value
        return f"[{left:.{digits}f}, {right:.{digits}f}]"
    return str(value)


def print_table(rows, columns=None, digits=6):
    if not rows:
        print("No rows to display.")
        return

    if columns is None:
        columns = list(rows[0].keys())

    widths = {}
    for column in columns:
        widths[column] = max(
            len(str(column)),
            max(len(format_value(row.get(column, ""), digits)) for row in rows),
        )

    header = " | ".join(str(column).ljust(widths[column]) for column in columns)
    separator = "-+-".join("-" * widths[column] for column in columns)

    print(header)
    print(separator)
    for row in rows:
        line = " | ".join(
            format_value(row.get(column, ""), digits).ljust(widths[column]) for column in columns
        )
        print(line)


def run_asian_assignment_cases():
    cases = [
        {"sigma": 0.3, "K": 100, "n": 50, "type": "put"},
        {"sigma": 0.3, "K": 100, "n": 100, "type": "put"},
        {"sigma": 0.4, "K": 100, "n": 50, "type": "put"},
        {"sigma": 0.3, "K": 100, "n": 50, "type": "call"},
        {"sigma": 0.3, "K": 100, "n": 100, "type": "call"},
        {"sigma": 0.4, "K": 100, "n": 50, "type": "call"},
    ]

    rows = []
    for idx, case in enumerate(cases):
        seed = DEFAULT_SEED + idx
        geometric_price = geometric_asian_option_price(
            100, case["sigma"], 0.05, 3, case["K"], case["n"], case["type"]
        )
        raw_mc = arithmetic_asian_option_mc(
            100,
            case["sigma"],
            0.05,
            3,
            case["K"],
            case["n"],
            case["type"],
            n_paths=100_000,
            control_variate="none",
            seed=seed,
        )
        cv_mc = arithmetic_asian_option_mc(
            100,
            case["sigma"],
            0.05,
            3,
            case["K"],
            case["n"],
            case["type"],
            n_paths=100_000,
            control_variate="geometric",
            seed=seed,
        )

        rows.append(
            {
                "type": case["type"],
                "sigma": case["sigma"],
                "K": case["K"],
                "n": case["n"],
                "geo_closed_form": geometric_price,
                "arith_mc_raw": raw_mc["price"],
                "raw_ci95": raw_mc["ci95"],
                "arith_mc_cv": cv_mc["price"],
                "cv_ci95": cv_mc["ci95"],
            }
        )
    return rows


def run_basket_assignment_cases():
    cases = [
        {"S1": 100, "S2": 100, "K": 100, "sigma1": 0.3, "sigma2": 0.3, "rho": 0.5, "type": "put"},
        {"S1": 100, "S2": 100, "K": 100, "sigma1": 0.3, "sigma2": 0.3, "rho": 0.9, "type": "put"},
        {"S1": 100, "S2": 100, "K": 100, "sigma1": 0.1, "sigma2": 0.3, "rho": 0.5, "type": "put"},
        {"S1": 100, "S2": 100, "K": 80, "sigma1": 0.3, "sigma2": 0.3, "rho": 0.5, "type": "put"},
        {"S1": 100, "S2": 100, "K": 120, "sigma1": 0.3, "sigma2": 0.3, "rho": 0.5, "type": "put"},
        {"S1": 100, "S2": 100, "K": 100, "sigma1": 0.5, "sigma2": 0.5, "rho": 0.5, "type": "put"},
        {"S1": 100, "S2": 100, "K": 100, "sigma1": 0.3, "sigma2": 0.3, "rho": 0.5, "type": "call"},
        {"S1": 100, "S2": 100, "K": 100, "sigma1": 0.3, "sigma2": 0.3, "rho": 0.9, "type": "call"},
        {"S1": 100, "S2": 100, "K": 100, "sigma1": 0.1, "sigma2": 0.3, "rho": 0.5, "type": "call"},
        {"S1": 100, "S2": 100, "K": 80, "sigma1": 0.3, "sigma2": 0.3, "rho": 0.5, "type": "call"},
        {"S1": 100, "S2": 100, "K": 120, "sigma1": 0.3, "sigma2": 0.3, "rho": 0.5, "type": "call"},
        {"S1": 100, "S2": 100, "K": 100, "sigma1": 0.5, "sigma2": 0.5, "rho": 0.5, "type": "call"},
    ]

    rows = []
    for idx, case in enumerate(cases):
        seed = DEFAULT_SEED + idx
        geometric_price = geometric_basket_option_price(
            case["S1"],
            case["S2"],
            case["sigma1"],
            case["sigma2"],
            0.05,
            3,
            case["K"],
            case["rho"],
            case["type"],
        )
        raw_mc = arithmetic_basket_option_mc(
            case["S1"],
            case["S2"],
            case["sigma1"],
            case["sigma2"],
            0.05,
            3,
            case["K"],
            case["rho"],
            case["type"],
            n_paths=100_000,
            control_variate="none",
            seed=seed,
        )
        cv_mc = arithmetic_basket_option_mc(
            case["S1"],
            case["S2"],
            case["sigma1"],
            case["sigma2"],
            0.05,
            3,
            case["K"],
            case["rho"],
            case["type"],
            n_paths=100_000,
            control_variate="geometric",
            seed=seed,
        )

        rows.append(
            {
                "type": case["type"],
                "K": case["K"],
                "sigma1": case["sigma1"],
                "sigma2": case["sigma2"],
                "rho": case["rho"],
                "geo_closed_form": geometric_price,
                "arith_mc_raw": raw_mc["price"],
                "raw_ci95": raw_mc["ci95"],
                "arith_mc_cv": cv_mc["price"],
                "cv_ci95": cv_mc["ci95"],
            }
        )
    return rows


## 8. Required Assignment Test Cases

这一节直接运行作业题目中明确给出的 Asian / Basket 参数集合。


In [ ]:
asian_rows = run_asian_assignment_cases()
print("Asian option test cases")
print_table(
    asian_rows,
    columns=[
        "type",
        "sigma",
        "K",
        "n",
        "geo_closed_form",
        "arith_mc_raw",
        "raw_ci95",
        "arith_mc_cv",
        "cv_ci95",
    ],
)


Asian option test cases
type | sigma    | K   | n   | geo_closed_form | arith_mc_raw | raw_ci95               | arith_mc_cv | cv_ci95               
-----+----------+-----+-----+-----------------+--------------+------------------------+-------------+-----------------------
put  | 0.300000 | 100 | 50  | 8.482705        | 7.737734     | [7.669065, 7.806402]   | 7.803611    | [7.799188, 7.808035]  
put  | 0.300000 | 100 | 100 | 8.431080        | 7.734220     | [7.665674, 7.802766]   | 7.749245    | [7.744831, 7.753658]  
put  | 0.400000 | 100 | 50  | 12.558769       | 11.240986    | [11.151196, 11.330777] | 11.282032   | [11.274196, 11.289867]
call | 0.300000 | 100 | 50  | 13.259126       | 14.594095    | [14.451350, 14.736840] | 14.733210   | [14.722201, 14.744219]
call | 0.300000 | 100 | 100 | 13.138779       | 14.647480    | [14.504571, 14.790389] | 14.611482   | [14.600457, 14.622507]
call | 0.400000 | 100 | 50  | 15.759820       | 18.104894    | [17.902610, 18.307177] | 18.208291   |

In [ ]:
basket_rows = run_basket_assignment_cases()
print("Basket option test cases")
print_table(
    basket_rows,
    columns=[
        "type",
        "K",
        "sigma1",
        "sigma2",
        "rho",
        "geo_closed_form",
        "arith_mc_raw",
        "raw_ci95",
        "arith_mc_cv",
        "cv_ci95",
    ],
)


Basket option test cases
type | K   | sigma1   | sigma2   | rho      | geo_closed_form | arith_mc_raw | raw_ci95               | arith_mc_cv | cv_ci95               
-----+-----+----------+----------+----------+-----------------+--------------+------------------------+-------------+-----------------------
put  | 100 | 0.300000 | 0.300000 | 0.500000 | 11.491573       | 10.607492    | [10.512764, 10.702219] | 10.574486   | [10.562392, 10.586580]
put  | 100 | 0.300000 | 0.300000 | 0.900000 | 12.622350       | 12.491501    | [12.386319, 12.596684] | 12.429362   | [12.426630, 12.432094]
put  | 100 | 0.100000 | 0.300000 | 0.500000 | 6.586381        | 5.522856     | [5.465666, 5.580047]   | 5.524062    | [5.515543, 5.532580]  
put  | 80  | 0.300000 | 0.300000 | 0.500000 | 4.711577        | 4.226970     | [4.171930, 4.282010]   | 4.245595    | [4.237830, 4.253361]  
put  | 120 | 0.300000 | 0.300000 | 0.500000 | 21.289105       | 19.954189    | [19.820275, 20.088102] | 19.879170   | [19.862892,

## 9. Additional Example Tests

题目允许其余类型自己设计测试。下面补充几个简洁的示例：

- 欧式期权价格与隐含波动率回推
- 美式期权二叉树
- KIKO put 的 quasi-MC 价格与 Delta


In [ ]:
sigma_true = 0.20
call_price = black_scholes_price(100, sigma_true, 0.05, 1.0, 100, "call", q=0.0)
put_price = black_scholes_price(100, sigma_true, 0.05, 1.0, 100, "put", q=0.0)
recovered_sigma = implied_volatility(call_price, 100, 0.05, 0.0, 1.0, 100, "call")

euro_rows = [
    {"metric": "European call price", "value": call_price},
    {"metric": "European put price", "value": put_price},
    {"metric": "Recovered implied vol", "value": recovered_sigma},
    {"metric": "American put (steps=200)", "value": american_option_binomial(100, 0.20, 0.05, 1.0, 100, 200, "put")},
    {"metric": "American call (steps=200)", "value": american_option_binomial(100, 0.20, 0.05, 1.0, 100, 200, "call")},
]

print("European / implied volatility / American examples")
print_table(euro_rows, columns=["metric", "value"])

kiko_result = quasi_mc_kiko_put(
    s0=100,
    sigma=0.20,
    r=0.05,
    T=3.0,
    K=100,
    L=80,
    U=130,
    n_obs=50,
    rebate=2.0,
    n_paths=8192,
    n_batches=8,
    seed=DEFAULT_SEED,
)

kiko_rows = [
    {"metric": "KIKO price", "value": kiko_result["price"]},
    {"metric": "KIKO price stderr", "value": kiko_result["price_stderr"]},
    {"metric": "KIKO delta", "value": kiko_result["delta"]},
    {"metric": "KIKO delta stderr", "value": kiko_result["delta_stderr"]},
    {"metric": "Effective QMC paths", "value": kiko_result["effective_paths"]},
    {"metric": "Spot bump used", "value": kiko_result["bump"]},
]

print()
print("KIKO quasi-Monte Carlo example")
print_table(kiko_rows, columns=["metric", "value"])


European / implied volatility / American examples
metric                    | value    
--------------------------+----------
European call price       | 10.450584
European put price        | 5.573526 
Recovered implied vol     | 0.200000 
American put (steps=200)  | 6.086383 
American call (steps=200) | 10.440591

KIKO quasi-Monte Carlo example
metric              | value    
--------------------+----------
KIKO price          | 6.908678 
KIKO price stderr   | 0.162860 
KIKO delta          | -0.245599
KIKO delta stderr   | 0.008633 
Effective QMC paths | 8192     
Spot bump used      | 1.000000 


## 10. Optional Notebook GUI

作业中要求一个 GUI。由于这份提交形式是 notebook，这里用 `ipywidgets` 做了一个轻量的交互界面：

- 先选择产品类型
- 再修改相关参数
- 点击 `Calculate` 输出结果

不相关的输入框会被忽略，因此同一个界面就可以覆盖全部模型。


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    product = widgets.Dropdown(
        options=[
            "European BS",
            "Implied Volatility",
            "Geometric Asian",
            "Arithmetic Asian MC",
            "Geometric Basket",
            "Arithmetic Basket MC",
            "KIKO Put QMC",
            "American Binomial",
        ],
        value="European BS",
        description="Product",
    )
    option_type = widgets.Dropdown(options=["call", "put"], value="call", description="Type")
    cv_choice = widgets.Dropdown(
        options=[("None", "none"), ("Geometric", "geometric")],
        value="none",
        description="CV",
    )

    s0 = widgets.FloatText(value=100.0, description="S0")
    s1 = widgets.FloatText(value=100.0, description="S1")
    s2 = widgets.FloatText(value=100.0, description="S2")
    sigma = widgets.FloatText(value=0.20, description="sigma")
    sigma1 = widgets.FloatText(value=0.30, description="sigma1")
    sigma2 = widgets.FloatText(value=0.30, description="sigma2")
    r = widgets.FloatText(value=0.05, description="r")
    q = widgets.FloatText(value=0.0, description="q")
    T = widgets.FloatText(value=1.0, description="T")
    K = widgets.FloatText(value=100.0, description="K")
    premium = widgets.FloatText(value=10.0, description="premium")
    rho = widgets.FloatText(value=0.5, description="rho")
    L = widgets.FloatText(value=80.0, description="L")
    U = widgets.FloatText(value=130.0, description="U")
    rebate = widgets.FloatText(value=2.0, description="rebate")
    n_obs = widgets.IntText(value=50, description="n_obs")
    steps = widgets.IntText(value=200, description="steps")
    n_paths = widgets.IntText(value=100000, description="n_paths")
    seed = widgets.IntText(value=42, description="seed")

    run_button = widgets.Button(description="Calculate", button_style="success")
    output = widgets.Output()


    def show_result(lines):
        for line in lines:
            print(line)


    def on_click(_):
        with output:
            output.clear_output()
            try:
                if product.value == "European BS":
                    price = black_scholes_price(s0.value, sigma.value, r.value, T.value, K.value, option_type.value, q.value)
                    show_result([f"Price: {price:.6f}"])

                elif product.value == "Implied Volatility":
                    iv = implied_volatility(premium.value, s0.value, r.value, q.value, T.value, K.value, option_type.value)
                    show_result([f"Implied volatility: {iv:.6f}"])

                elif product.value == "Geometric Asian":
                    price = geometric_asian_option_price(s0.value, sigma.value, r.value, T.value, K.value, n_obs.value, option_type.value)
                    show_result([f"Geometric Asian price: {price:.6f}"])

                elif product.value == "Arithmetic Asian MC":
                    result = arithmetic_asian_option_mc(
                        s0.value,
                        sigma.value,
                        r.value,
                        T.value,
                        K.value,
                        n_obs.value,
                        option_type.value,
                        n_paths=n_paths.value,
                        control_variate=cv_choice.value,
                        seed=seed.value,
                    )
                    show_result(
                        [
                            f"Arithmetic Asian price: {result['price']:.6f}",
                            f"95% CI: [{result['ci95'][0]:.6f}, {result['ci95'][1]:.6f}]",
                            f"Beta: {result['beta']:.6f}",
                        ]
                    )

                elif product.value == "Geometric Basket":
                    price = geometric_basket_option_price(
                        s1.value, s2.value, sigma1.value, sigma2.value, r.value, T.value, K.value, rho.value, option_type.value
                    )
                    show_result([f"Geometric Basket price: {price:.6f}"])

                elif product.value == "Arithmetic Basket MC":
                    result = arithmetic_basket_option_mc(
                        s1.value,
                        s2.value,
                        sigma1.value,
                        sigma2.value,
                        r.value,
                        T.value,
                        K.value,
                        rho.value,
                        option_type.value,
                        n_paths=n_paths.value,
                        control_variate=cv_choice.value,
                        seed=seed.value,
                    )
                    show_result(
                        [
                            f"Arithmetic Basket price: {result['price']:.6f}",
                            f"95% CI: [{result['ci95'][0]:.6f}, {result['ci95'][1]:.6f}]",
                            f"Beta: {result['beta']:.6f}",
                        ]
                    )

                elif product.value == "KIKO Put QMC":
                    result = quasi_mc_kiko_put(
                        s0=s0.value,
                        sigma=sigma.value,
                        r=r.value,
                        T=T.value,
                        K=K.value,
                        L=L.value,
                        U=U.value,
                        n_obs=n_obs.value,
                        rebate=rebate.value,
                        n_paths=max(1024, n_paths.value // 10),
                        n_batches=8,
                        seed=seed.value,
                    )
                    show_result(
                        [
                            f"KIKO price: {result['price']:.6f}",
                            f"KIKO delta: {result['delta']:.6f}",
                            f"Price stderr: {result['price_stderr']:.6f}",
                            f"Delta stderr: {result['delta_stderr']:.6f}",
                        ]
                    )

                elif product.value == "American Binomial":
                    price = american_option_binomial(
                        s0.value, sigma.value, r.value, T.value, K.value, steps.value, option_type.value, q=q.value
                    )
                    show_result([f"American option price: {price:.6f}"])

            except Exception as exc:
                print(f"Error: {exc}")


    run_button.on_click(on_click)

    control_rows = [
        widgets.HBox([product, option_type, cv_choice]),
        widgets.HBox([s0, s1, s2]),
        widgets.HBox([sigma, sigma1, sigma2]),
        widgets.HBox([r, q, T, K]),
        widgets.HBox([premium, rho, L, U]),
        widgets.HBox([rebate, n_obs, steps, n_paths]),
        widgets.HBox([seed]),
    ]

    display(widgets.VBox(control_rows + [run_button, output]))

except ImportError:
    print("ipywidgets is not installed in the current environment.")
